# ChemBreak Candidate Task Generator

This notebook runs an open-weight LLM **inside Google Colab**. It does not call a paid LLM API.

Default model: `Qwen/Qwen3-4B-Instruct-2507`

The first supplied run is a 50-candidate pilot. After checking quality, edit `run_config.json` to scale up.


In [ ]:
# 1. Confirm that Colab has a GPU.
import torch, platform
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
else:
    raise RuntimeError("No GPU detected. Choose Runtime > Change runtime type > GPU, then rerun.")


In [ ]:
# 2. Mount Google Drive.
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
# 3. Set the project folder.
# Upload the extracted ChemBreak_Colab_Generator_v1 folder to the root of My Drive.
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/ChemBreak_Colab_Generator_v1")
assert PROJECT_DIR.exists(), f"Project folder not found: {PROJECT_DIR}"

print("Project directory:", PROJECT_DIR)
print("\nFiles:")
for p in sorted(PROJECT_DIR.iterdir()):
    print(" -", p.name)


In [ ]:
# 4. Install the local-generation dependencies.
!pip -q install -r "/content/drive/MyDrive/ChemBreak_Colab_Generator_v1/requirements_colab.txt"


## Restart note

If Colab tells you that recently installed packages require a runtime restart, use **Runtime > Restart session**, then rerun the notebook from the top. Usually the current Colab image can continue directly.


In [ ]:
# 5. Import the ChemBreak local generator.
import sys, json, importlib
sys.path.insert(0, str(PROJECT_DIR))

import generate_local
importlib.reload(generate_local)

CONFIG_PATH = PROJECT_DIR / "run_config.json"
config = generate_local.load_config(CONFIG_PATH)

print(json.dumps(config, indent=2))


In [ ]:
# 6. Inspect the generation matrix before loading the model.
matrix_path = PROJECT_DIR / config["matrix_file"]
matrix = generate_local.load_matrix(matrix_path)
selected = generate_local.select_rows(matrix, config)

print("Total matrix rows:", len(matrix))
print("Rows selected by current config:", len(selected))
display(selected.head(10))


In [ ]:
# 7. Dry-run the exact prompt for the first selected matrix row.
template = generate_local.load_prompt_template(PROJECT_DIR / config["prompt_file"])
row = selected.iloc[0]
n = int(config.get("n_per_row") or row["DEFAULT_N_CANDIDATES"])
n = min(n, int(config.get("call_batch_size", 5)))

rendered = generate_local.render_prompt(template, row, n)
print(rendered)


## Load the open-weight model

This is the step that downloads the model from Hugging Face and places it in the Colab runtime.

No paid LLM API key is used.

The default configuration enables 4-bit loading to reduce GPU memory use.


In [ ]:
# 8. Load the model locally into the Colab GPU.
tokenizer, model = generate_local.load_local_model(
    config["model_id"],
    load_in_4bit=bool(config.get("load_in_4bit", True)),
    cache_dir=config.get("hf_cache_dir") or None,
)


In [ ]:
# 9. Optional single-batch test before running multiple matrix rows.
# This does not save anything. It lets you see whether the model returns valid JSON.
test_n = min(2, int(config.get("call_batch_size", 5)))
test_prompt = generate_local.render_prompt(template, row, test_n)

raw = generate_local.generate_text(
    tokenizer,
    model,
    test_prompt,
    temperature=float(config.get("temperature", 0.85)),
    top_p=float(config.get("top_p", 0.95)),
    repetition_penalty=float(config.get("repetition_penalty", 1.05)),
    max_new_tokens=int(config.get("max_new_tokens", 2600)),
    seed=int(config.get("seed", 42)),
)

print(raw)


In [ ]:
# 10. Validate the test response.
parsed = generate_local.parse_json_response(raw)
validated = generate_local.validate_response(
    parsed,
    expected_n=test_n,
    allowed_scenarios=generate_local.split_scenarios(row["ALLOWED_SCENARIOS"]),
)
print("Validated candidates:", len(validated))
for i, item in enumerate(validated, 1):
    print(f"\nCandidate {i}")
    print(item["benchmark_prompt"])


## Generate and checkpoint

The next cell performs the actual row-by-row generation.

Output is appended directly to `candidate_tasks.csv` in Google Drive after every valid batch. If Colab disconnects, reconnect and rerun. Existing candidate IDs are skipped because `resume` is enabled.


In [ ]:
# 11. Run the configured generation job.
output_path = generate_local.run_generation(
    PROJECT_DIR,
    config,
    tokenizer,
    model,
)


In [ ]:
# 12. Review progress and the generated candidate table.
import pandas as pd

generate_local.print_candidate_summary(output_path)

candidates = pd.read_csv(output_path)
print("\nLatest rows:")
display(candidates.tail(20))

progress_path = PROJECT_DIR / config["progress_file"]
if progress_path.exists():
    print("\nGeneration progress:")
    display(pd.read_csv(progress_path).tail(30))


## Scaling after the pilot

The supplied config targets 10 rows × 5 candidates = 50 candidates.

After reviewing those tasks, edit `run_config.json`.

For 25 candidates across all 271 matrix rows:

```json
"n_per_row": 25,
"start_row": 1,
"end_row": 271
```

That targets 6,775 raw candidates.

For 50 candidates per row, set `"n_per_row": 50`, which targets 13,550 raw candidates.

Keep `"resume": true`.


In [ ]:
# 13. Optional GPU memory report.
if torch.cuda.is_available():
    print("Allocated GB:", round(torch.cuda.memory_allocated() / 1024**3, 2))
    print("Reserved GB:", round(torch.cuda.memory_reserved() / 1024**3, 2))
